# Helio Mirage

This notebook is designed to run the Task B (Diagnosing Object Hallucination) and Task A (Filament Segmentation) training/evaluations directly on Google Colab, circumventing local hardware constraints (VRAM and storage).

## Setup Instructions
1. Upload the entire `helio-mirage` folder from your local machine to your Google Drive.
2. Ensure `filament-segmentation-2026.zip` is in the root of your Google Drive (`/content/drive/MyDrive/`).
3. Set your Hugging Face Token (`HF_TOKEN`) and Weights & Biases API Key (`WANDB_API_KEY`) below.

In [44]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Change directory to the uploaded project folder in your Drive
%cd /content/drive/MyDrive/helio-mirage

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/helio-mirage


In [ ]:
import os

# Unzip the dataset directly into the correct folder
# It is much faster to let Colab unzip it than to unzip it manually in Drive
dataset_zip_path = "/content/drive/MyDrive/filament-segmentation-2026.zip"
extract_target_dir = "data/raw/MAGFiLO"

if not os.path.exists(extract_target_dir):
    os.makedirs(extract_target_dir, exist_ok=True)
    print("Extracting dataset... this might take a few minutes.")
    !unzip -q "{dataset_zip_path}" -d "{extract_target_dir}"
    print("Extraction complete!")
else:
    print("Dataset folder already exists. Skipping extraction.")

Extracting dataset... this might take a few minutes.
Extraction complete!


In [ ]:
!find data/raw -type f | head -n 10


data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/test/test_images/20110120105534Ch.jpeg
data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/test/test_images/20110130110334Ch.jpeg
data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/test/test_images/20110214175414Mh.jpeg
data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/test/test_images/20110329082654Uh.jpeg
data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/test/test_images/20110408083214Th.jpeg
data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/test/test_images/20110424064534Lh.jpeg
data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/test/test_images/20110429120534Ch.jpeg
data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/test/test_images/20110529063154Uh.jpeg
data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/test/test_images/20110531070514Th.jpeg
data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/test/test_images/20110603123034Ch.jpeg


In [ ]:
import os
import glob

# Check the training split
base_path = "data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/train"

# Get just the filenames (without the full folder path)
images = set([os.path.basename(p) for p in glob.glob(os.path.join(base_path, "train_images", "*.*"))])
masks = set([os.path.basename(p) for p in glob.glob(os.path.join(base_path, "train_masks", "*.*"))])
spines = set([os.path.basename(p) for p in glob.glob(os.path.join(base_path, "train_spines", "*.*"))])

print(f"Total images: {len(images)}")
print(f"Total masks: {len(masks)}")
print(f"Total spines: {len(spines)}")

# Find the differences
missing_masks = images - masks
missing_spines = images - spines

print(f"\nImages missing masks: {len(missing_masks)}")
if len(missing_masks) > 0:
    print(f"Examples: {list(missing_masks)[:5]}")

print(f"\nImages missing spines: {len(missing_spines)}")
if len(missing_spines) > 0:
    print(f"Examples: {list(missing_spines)[:5]}")


Total images: 707
Total masks: 0
Total spines: 0

Images missing masks: 707
Examples: ['20171025133730Ch.jpeg', '20160402025254Uh.jpeg', '20160804133734Ch.jpeg', '20130324015554Uh.jpeg', '20170809164910Mh.jpeg']

Images missing spines: 707
Examples: ['20171025133730Ch.jpeg', '20160402025254Uh.jpeg', '20160804133734Ch.jpeg', '20130324015554Uh.jpeg', '20170809164910Mh.jpeg']


In [ ]:
!ls data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/train


MAGFiLO_1.0_Annotations_kaggle2026_train.json  train_images


In [ ]:
!head -c 1000 data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json


{
    "info": {
        "description": "MLEcoFi 2024 Dataset - V1.0",
        "url": "https://www.mlecofi.net/",
        "version": "1.0",
        "year": 2024,
        "contributor": "MLEcoFi Team",
        "date_created": "2024-07-15",
        "team": "Earth-Space AI Research Lab (www.esairlab.com)",
        "collaborators": [
            "National Solar Observatory, Georgia State University"
        ],
        "date_updated": "2026-06-01"
    },
    "licenses": {
        "url": "https://gong.nso.edu/",
        "id": 1,
        "name": "NSO/GONG"
    },
    "categories": [
        {
            "supercategory": "filament",
            "id": 1,
            "name": "Left"
        },
        {
            "supercategory": "filament",
            "id": 2,
            "name": "Right"
        },
        {
            "supercategory": "filament",
            "id": 3,
            "name": "Unidentifiable"
        },
        {
            "supercategory": "filament",
            "id": 4,
     

In [ ]:
os.environ['HF_TOKEN']="use_ur_own"
os.environ['WANDB_API_KEY']='use_ur_own'


In [ ]:
# Install dependencies
!pip install -r requirements.txt
!pip install wandb transformers accelerate peft bitsandbytes pycocotools opencv-python-headless

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## Task A: Train Filament Segmentation Model


In [ ]:
!PYTHONPATH=. python src/task_a/scripts/prepare_magfilo.py

Loading annotations for train from data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json...
loading annotations into memory...
Done (t=0.73s)
creating index...
index created!
Found 1154 images in train. Rasterizing masks and spines...
100% 1154/1154 [01:01<00:00, 18.80it/s]
Finished processing train. Masks saved to data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/train/train_masks.
Skipping test - no annotations found at data/raw/MAGFiLO/MAGFiLO_1.0_Kaggle_2026/test/MAGFiLO_1.0_Annotations_kaggle2026_test.json


In [ ]:
!PYTHONPATH=. python src/task_a/train.py

Using device: cuda
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch 1:   0% 0/177 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_r

## Task B: Run Diagnostics Baseline
This runs the zero-shot evaluation on the POPE benchmark.

In [ ]:
!PYTHONPATH=. python src/task_b/eval/pope_eval.py --model_id liuhaotian/llava-v1.5-7b --split all --use_4bit

Starting POPE Evaluation for liuhaotian/llava-v1.5-7b
Split: all, 4-bit: True
Loading model...
Running evaluation (this is a placeholder for actual VLM inference)...

--- POPE Results ---
Random - Accuracy: 85%, FPR: 10%, Yes/No Bias: 1.1
Popular - Accuracy: 80%, FPR: 15%, Yes/No Bias: 1.2
Adversarial - Accuracy: 75%, FPR: 25%, Yes/No Bias: 1.5


## Task B: Run LoRA Fine-Tuning
This runs the parameter-efficient fine-tuning.

In [ ]:
!PYTHONPATH=. python src/task_b/train.py --config-name lora_config.yaml

Starting LoRA fine-tuning with config: lora_config.yaml
Initializing PEFT configuration...
Loading contrastive dataset...
Starting training loop (Placeholder)...
Epoch 1/3 - Loss: 1.5000
Epoch 2/3 - Loss: 0.7500
Epoch 3/3 - Loss: 0.5000
Training complete! Model saved to checkpoints/
